# 01 · SQL practice (runs locally with SQLite, nothing to install)

10 interview-style questions on a small clinic database: JOIN, GROUP BY/HAVING, subqueries, CTEs and window
functions. **Write your query in the empty cell first.** Solutions are at the bottom.

In [1]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")
con.executescript('''
CREATE TABLE patients (id INTEGER PRIMARY KEY, name TEXT, sex TEXT, birth_year INT, city TEXT);
CREATE TABLE doctors  (id INTEGER PRIMARY KEY, name TEXT, specialty TEXT);
CREATE TABLE visits   (id INTEGER PRIMARY KEY, patient_id INT, doctor_id INT, visit_date TEXT, fee REAL, diagnosis TEXT);
INSERT INTO patients VALUES
 (1,'Rahim','M',1980,'Dhaka'),(2,'Karima','F',1992,'Dhaka'),(3,'Sumon','M',1975,'Chattogram'),
 (4,'Nadia','F',2001,'Sylhet'),(5,'Arif','M',1968,'Dhaka'),(6,'Tania','F',1988,'Khulna');
INSERT INTO doctors VALUES (1,'Dr. Hasan','Cardiology'),(2,'Dr. Rina','Medicine'),(3,'Dr. Kabir','Pediatrics');
INSERT INTO visits VALUES
 (1,1,1,'2026-01-05',1500,'hypertension'),(2,1,1,'2026-02-10',1500,'hypertension'),
 (3,2,2,'2026-01-20',800,'fever'),(4,3,1,'2026-03-02',1500,'angina'),(5,3,2,'2026-03-15',800,'diabetes'),
 (6,5,1,'2026-01-11',1500,'angina'),(7,5,1,'2026-04-01',1500,'angina'),(8,5,2,'2026-04-20',800,'diabetes'),
 (9,4,3,'2026-02-14',1000,'asthma'),(10,2,2,'2026-05-03',800,'fever');
''')
q = lambda sql: pd.read_sql_query(sql, con)
q("SELECT * FROM visits LIMIT 3")

,id,patient_id,doctor_id,visit_date,fee,diagnosis
0,1,1,1,2026-01-05,1500.0,hypertension
1,2,1,1,2026-02-10,1500.0,hypertension
2,3,2,2,2026-01-20,800.0,fever


| # | Question |
|---|---|
| 1 | Total fee collected per doctor (name, total), highest first |
| 2 | Patients with no visits at all (LEFT JOIN) |
| 3 | Cities with more than one patient (GROUP BY + HAVING) |
| 4 | Each patient's number of visits and total spend, including patients with zero visits |
| 5 | The most recent visit of each patient (window function `ROW_NUMBER`) |
| 6 | Running total of revenue by visit date (window `SUM ... OVER`) |
| 7 | Patients who visited **both** Cardiology and Medicine |
| 8 | Monthly revenue (`strftime('%Y-%m', visit_date)`) |
| 9 | Rank doctors by number of distinct patients (`DENSE_RANK`) |
| 10 | Days between each patient's consecutive visits (`LAG` + `julianday`) |

In [2]:
# Q1: your query
q('''
SELECT 1
''')

,1
0,1


In [3]:
# Q2: your query

In [4]:
# Q3: your query

In [5]:
# Q4: your query

In [6]:
# Q5: your query

In [7]:
# Q6..Q10: your queries

## Solutions (check after trying)

In [8]:
solutions = {
 1: '''SELECT d.name, SUM(v.fee) AS total FROM visits v JOIN doctors d ON d.id = v.doctor_id
       GROUP BY d.id ORDER BY total DESC''',
 2: '''SELECT p.name FROM patients p LEFT JOIN visits v ON v.patient_id = p.id WHERE v.id IS NULL''',
 3: '''SELECT city, COUNT(*) AS n FROM patients GROUP BY city HAVING COUNT(*) > 1''',
 4: '''SELECT p.name, COUNT(v.id) AS visits, COALESCE(SUM(v.fee), 0) AS spend
       FROM patients p LEFT JOIN visits v ON v.patient_id = p.id GROUP BY p.id ORDER BY spend DESC''',
 5: '''SELECT name, visit_date, diagnosis FROM (
         SELECT p.name, v.visit_date, v.diagnosis,
                ROW_NUMBER() OVER (PARTITION BY v.patient_id ORDER BY v.visit_date DESC) AS rn
         FROM visits v JOIN patients p ON p.id = v.patient_id) WHERE rn = 1''',
 6: '''SELECT visit_date, fee, SUM(fee) OVER (ORDER BY visit_date) AS running_total FROM visits''',
 7: '''SELECT p.name FROM visits v JOIN doctors d ON d.id = v.doctor_id JOIN patients p ON p.id = v.patient_id
       WHERE d.specialty IN ('Cardiology','Medicine') GROUP BY p.id HAVING COUNT(DISTINCT d.specialty) = 2''',
 8: '''SELECT strftime('%Y-%m', visit_date) AS month, SUM(fee) AS revenue FROM visits GROUP BY month ORDER BY month''',
 9: '''WITH c AS (SELECT doctor_id, COUNT(DISTINCT patient_id) AS n FROM visits GROUP BY doctor_id)
       SELECT d.name, c.n, DENSE_RANK() OVER (ORDER BY c.n DESC) AS rnk FROM c JOIN doctors d ON d.id = c.doctor_id''',
 10: '''SELECT patient_id, visit_date,
          julianday(visit_date) - julianday(LAG(visit_date) OVER (PARTITION BY patient_id ORDER BY visit_date)) AS gap_days
        FROM visits ORDER BY patient_id, visit_date''',
}
for k, sql in solutions.items():
    print(f"--- Q{k}"); print(q(sql).to_string(index=False))

--- Q1
     name  total
Dr. Hasan 7500.0
 Dr. Rina 3200.0
Dr. Kabir 1000.0
--- Q2
 name
Tania
--- Q3
 city  n
Dhaka  3
--- Q4
  name  visits  spend
  Arif       3 3800.0
 Rahim       2 3000.0
 Sumon       2 2300.0
Karima       2 1600.0
 Nadia       1 1000.0
 Tania       0    0.0
--- Q5
  name visit_date    diagnosis
 Rahim 2026-02-10 hypertension
Karima 2026-05-03        fever
 Sumon 2026-03-15     diabetes
 Nadia 2026-02-14       asthma
  Arif 2026-04-20     diabetes
--- Q6
visit_date    fee  running_total
2026-01-05 1500.0         1500.0
2026-01-11 1500.0         3000.0
2026-01-20  800.0         3800.0
2026-02-10 1500.0         5300.0
2026-02-14 1000.0         6300.0
2026-03-02 1500.0         7800.0
2026-03-15  800.0         8600.0
2026-04-01 1500.0        10100.0
2026-04-20  800.0        10900.0
2026-05-03  800.0        11700.0
--- Q7
 name
Sumon
 Arif
--- Q8
  month  revenue
2026-01   3800.0
2026-02   2500.0
2026-03   2300.0
2026-04   2300.0
2026-05    800.0
--- Q9
     name  n  rn

## pandas equivalents (often asked in the same interview)

In [9]:
v, p, d = q("SELECT * FROM visits"), q("SELECT * FROM patients"), q("SELECT * FROM doctors")
print(v.merge(d, left_on="doctor_id", right_on="id").groupby("name").fee.sum().sort_values(ascending=False))   # Q1
v["rn"] = v.sort_values("visit_date", ascending=False).groupby("patient_id").cumcount() + 1                       # Q5
print(v[v.rn == 1][["patient_id", "visit_date", "diagnosis"]])

name
Dr. Hasan    7500.0
Dr. Rina     3200.0
Dr. Kabir    1000.0
Name: fee, dtype: float64
   patient_id  visit_date     diagnosis
1           1  2026-02-10  hypertension
4           3  2026-03-15      diabetes
7           5  2026-04-20      diabetes
8           4  2026-02-14        asthma
9           2  2026-05-03         fever
